# API 2 - MOD-4 Vasquez Jorge Luis

### Actividad 2 - Procesamiento Natural del Lenguaje (NPL)

Este notebook contiene la solución a la actividad 2. El objetivo es realizar preprocesamiento de texto sobre un corpus de comentarios, incluyendo limpieza, tokenización, lematización, stemming, split en entrenamiento y prueba, y análisis de la distribución del target.

In [ ]:
# Celda 1: Importación de librerías necesarias
import sys
import re
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

# Intentar importar nltk y spaCy, instalarlos si no existen
try:
    import nltk
except ImportError:
    !{sys.executable} -m pip install -q nltk
    import nltk

try:
    from nltk.tokenize import RegexpTokenizer
    from nltk.corpus import stopwords
    from nltk.stem.snowball import SnowballStemmer
except ImportError:
    !{sys.executable} -m pip install -q nltk
    import nltk
    from nltk.tokenize import RegexpTokenizer
    from nltk.corpus import stopwords
    from nltk.stem.snowball import SnowballStemmer

nltk.download("punkt")
nltk.download("stopwords")

try:
    import spacy
    nlp = spacy.load("es_core_news_sm")
except Exception:
    try:
        import spacy
    except ImportError:
        !{sys.executable} -m pip install -q spacy
        import spacy
    try:
        nlp = spacy.load("es_core_news_sm")
    except Exception:
        !{sys.executable} -m spacy download es_core_news_sm
        nlp = spacy.load("es_core_news_sm")

plt.rcParams["figure.figsize"] = (10, 5)
sns.set_theme(style="whitegrid")


In [ ]:
# Celda 2: Cargar el conjunto de datos de comentarios
comentarios = pd.read_csv("comentarios.csv")
comentarios.head(10)


In [ ]:
# Vista general del dataset
print("Shape del dataset:", comentarios.shape)
print("\nDistribución del target:")
print(comentarios["tipo"].value_counts(normalize=True).mul(100).round(2))
comentarios.info()


## 1. Preprocesamiento de datos

A continuación se define la función de limpieza y tokenización usando `RegexpTokenizer`. El patrón utilizado captura secuencias de letras españolas, incluidos acentos y la letra ñ. Esto permite extraer tokens limpios y separar signos de puntuación, emojis y números.

In [ ]:
# Celda 3: Definir funciones de limpieza y tokenización
patron_token = r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+"
regex_tokenizer = RegexpTokenizer(patron_token)
spanish_stopwords = set(stopwords.words("spanish"))

def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).lower()
    texto = re.sub(r"http\S+|www\S+", " ", texto)
    texto = re.sub(r"[^\w\sÁÉÍÓÚÜÑáéíóúüñ]", " ", texto)
    texto = re.sub(r"\d+", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def preprocesar_texto(texto, tokenizer=regex_tokenizer, stopwords_set=spanish_stopwords):
    texto_limpio = limpiar_texto(texto)
    tokens = tokenizer.tokenize(texto_limpio)
    tokens = [token for token in tokens if token not in stopwords_set and len(token) > 2]
    return tokens

for ejemplo in comentarios["comentario"].head(5):
    print(ejemplo, "->", preprocesar_texto(ejemplo))


In [ ]:
# Celda 4: Aplicar preprocesamiento y crear nuevas columnas
comentarios["tokens_limpios"] = comentarios["comentario"].apply(preprocesar_texto)
comentarios[["comentario", "tokens_limpios"]].head(10)


## 2. Lematización y Stemming

Se definen dos funciones separadas para reducir el vocabulario. La función de lematización utiliza spaCy cuando está disponible y, en caso contrario, una estrategia de normalización basada en reglas. La función de stemming usa `SnowballStemmer` en español.

In [ ]:
# Celda 5: Definir lematización y stemming
spanish_stemmer = SnowballStemmer("spanish")

def lematizar_tokens(tokens):
    if "nlp" in globals():
        documento = nlp(" ".join(tokens))
        return [token.lemma_ for token in documento if token.lemma_ != ""]
    reglas = [r"(aciones|ación)$", r"(antes|encia|encias)$", r"(mente)$", r"(es)$", r"(s)$"]
    lemas = []
    for token in tokens:
        lema = token
        for regla in reglas:
            lema = re.sub(regla, "", lema)
        lemas.append(lema)
    return lemas

def stemizar_tokens(tokens):
    return [spanish_stemmer.stem(token) for token in tokens]

ejemplo = comentarios.loc[0, "tokens_limpios"]
print("Tokens originales:", ejemplo)
print("Lemas:", lematizar_tokens(ejemplo))
print("Raíces:", stemizar_tokens(ejemplo))


In [ ]:
# Celda 6: Crear corpus reducido a lemas y raíces
comentarios["tokens_lemas"] = comentarios["tokens_limpios"].apply(lematizar_tokens)
comentarios["tokens_raices"] = comentarios["tokens_limpios"].apply(stemizar_tokens)
comentarios[["comentario", "tokens_limpios", "tokens_lemas", "tokens_raices"]].head(10)


## 3. División de la muestra en entrenamiento y test

Se utiliza `train_test_split` de `sklearn` para separar el 80 % de los datos para entrenamiento y el 20 % para prueba. Se busca que la muestra test conserve el orden relativo de las clases, aunque con pocos datos la distribución puede variar.

In [ ]:
# Celda 7: Split en entrenamiento y prueba
train_df, test_df = train_test_split(comentarios, test_size=0.20, random_state=42, stratify=None)
print("Tamaño total:", comentarios.shape)
print("Tamaño train:", train_df.shape)
print("Tamaño test:", test_df.shape)


In [ ]:
# Celda 8: Distribución de target
def distribucion_porcentaje(df, etiqueta):
    return df["tipo"].value_counts(normalize=True).mul(100).round(2).rename(etiqueta)

dist_total = distribucion_porcentaje(comentarios, "Total")
dist_train = distribucion_porcentaje(train_df, "Train")
dist_test = distribucion_porcentaje(test_df, "Test")

print(dist_total.to_frame())
print("\n", dist_train.to_frame())
print("\n", dist_test.to_frame())


## 4. Distribución del target en gráficos

A continuación se visualizan las proporciones de cada clase en el conjunto total, de entrenamiento y de prueba. Esto ayuda a verificar si la muestra respeta al menos el orden de las clases.

In [ ]:
# Celda 9: Gráfico de barras de la distribución del target
plot_data = pd.DataFrame({
    "Total": dist_total,
    "Train": dist_train,
    "Test": dist_test
}).fillna(0).T

plot_data.plot(kind="bar", rot=0, figsize=(10, 6))
plt.title("Distribución porcentual del target por muestra")
plt.xlabel("Conjunto")
plt.ylabel("Porcentaje")
plt.legend(title="Tipo", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Conclusión

Se realizó el preprocesamiento de texto completo: selección e importación de librerías, limpieza y tokenización con `RegexpTokenizer`, reducción del vocabulario mediante lematización y stemming, y separación en conjuntos de entrenamiento y prueba. Además, se representó la distribución de las clases `bueno`, `malo` e `info` para comparar el total, train y test.